# Engine: The Archimedes Screw — prime coordinates on the axis u = ln x

**File:** `ValaQuenta/modules/archimedes_screw/`
**ValaQuenta wiki:** [wiki/archimedes_screw.md](../../wiki/archimedes_screw.md)
**Ainulindale wiki:** `Ainulindale/wiki/83_the_archimedes_screw.md`

∅_RB (formerly Ĥ_RB) is the **water** — the medium, the rest state, e₀. It is not
the engine. The engine is the **screw**: the helix that converts *rotation* into
*lift*, one quantised pitch per turn, and runs backward as a turbine.

That machine is the logarithm:

    log(p·q) = log p + log q

Multiplication on the wheel becomes addition on the tower. Working axis:

    u = ln x

Cody's four search terms — **Ordinal Value**, **Zeta Index Value**, **Number of
Digits**, **Total Spaces Between** — are four coordinates on that one axis, bound
by the von Mangoldt explicit formula:

    ψ(eᵘ) = eᵘ − 2e^(u/2) Σₖ cos(γₖu − arg ρₖ)/|ρₖ| − ln2π − ½ln(1 − e^(−2u))

Three facts fall out, and the notebook exhibits each:

1. every zero γₖ is a **tone** of frequency γₖ in u — the zeta index *is* the
   summation index;
2. ψ **jumps by exactly ln p** at u = ln p — the leaf-drop magnitude *is* the
   prime, not an encoding of it;
3. every tone shares the envelope 2√x **because** every ρ has real part ½ — that
   equal-envelope condition is RH stated in the prime domain.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))
import math
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

from ValaQuenta.modules.archimedes_screw import (
    lambert_w, screw_pitch, u_axis, digits_of, li, prime_count_log10,
    nth_prime_estimate, zero_count_smooth, zero_height_lambert, zero_height,
    zeros_upto, mean_gap, total_spaces, von_mangoldt, chebyshev_psi_exact,
    chebyshev_psi_explicit, leaf_drops, tone, interference_profile,
    amplitude_envelope, envelope_ratio, splitting_type, splitting_vector,
    ramified_primes, screw_coordinates, shake_order, OMEGA_ZS, ZEROS_KNOWN,
)
print('python', sys.version.split()[0], '| zeros tabulated:', len(ZEROS_KNOWN))

---
## 1. The screw is the logarithm — and its gear ratio is Lambert W

`lambert_w` solves W(x)·e^{W(x)} = x. Its fixed point W(1) = Ω_ZΣ is already
canonical in this project (`~/.clauderc`, `OMEGA_ZS`), where it pins σ = ½
(PAPER.md §12.1).

The point recorded here is that the **same** W also inverts the zero-counting
function to give the zero *heights*. W supplies both coordinates of every zero:
the real part through its fixed point, the imaginary part through its inverse.

In [ ]:
print(f"W(1)          = {lambert_w(1.0):.13f}")
print(f"OMEGA_ZS      = {OMEGA_ZS:.13f}")
print(f"identical     : {abs(lambert_w(1.0) - OMEGA_ZS) < 1e-12}")
print()
print("One turn of the screw = one prime = a lift of ln p:")
for p in (2, 3, 5, 7, 97, 1_000_003):
    print(f"  pitch({p:>9}) = {screw_pitch(p):.6f}")

---
## 2. The Lambert inverse of the zero count

Exact algebra on the smooth Riemann–von Mangoldt count, no fitting:

    N(T) = (T/2π)·ln(T/2πe) + 7/8
    set N(T) = n, T = 2πv   →   v·ln(v/e) = n
    (v/e)·ln(v/e) = n/e     →   ln(v/e) = W(n/e)
    ⇒  γₙ ≈ 2πn / W(n/e)

Asymptotic — S(T) is O(ln T) and dominates at small n, so the module prefers
the 50 tabulated LMFDB zeros below index 50 and switches to the closed form
above. Compare the two where both are available:

In [ ]:
print(f"{'n':>4} {'tabulated γₙ':>14} {'2πn/W(n/e)':>14} {'rel err':>10}")
for n in (1, 2, 5, 10, 20, 30, 40, 50):
    t = zero_height(n)
    w = zero_height_lambert(n)
    print(f"{n:>4} {t:>14.6f} {w:>14.6f} {abs(w-t)/t:>9.2%}")
print()
print("Beyond the table the closed form carries it:")
for n in (100, 1_000, 10_000, 10**6):
    print(f"  γ_{n:<8} ≈ {zero_height_lambert(n):,.3f}")

---
## 3. The search-term interface

Enter on any one of the four coordinates; leave on all of them. Everything
routes through u = ln x.

In [ ]:
def show(term, value):
    c = screw_coordinates(term, value)
    print(f"── entered on {term} = {value}")
    for k in ('magnitude', 'u', 'digits', 'ordinal', 'zeta_index',
              'mean_gap', 'zero_height'):
        v = c[k]
        print(f"     {k:<12} {v:>22,.4f}" if isinstance(v, float) else f"     {k:<12} {v}")
    print()

show('magnitude', 1_000_000)
show('digits', 10)
show('ordinal', 1000)
show('zeta_index', 25)

### Finiteness at RSA scale

The magnitude coordinate works in log space, so it does not overflow where the
numbers actually live. This is Cody's finiteness point, computed rather than
asserted: the candidate set is enormous and **finite**.

In [ ]:
for d in (10, 100, 309, 617):
    lg = prime_count_log10(d)
    print(f"  π(10^{d:<4}) ≈ 10^{lg:.2f}  =  2^{lg*math.log2(10):.1f}")
print()
print("  RSA-2048 modulus ≈ 10^617; each factor ≈ 10^309.")
print("  Candidate primes per factor ≈ 2^1017.  Finite. Structured. Countable.")

---
## 4. The binding equation: ψ rebuilt from tones

ψ(x) = Σ_{pᵐ ≤ x} ln p is the ground truth (exact, by sieve). The explicit
formula reconstructs it from the zeros. Truncating the zero sum at K terms
leaves an error controlled by ~x/K — **this is the resolution wall**, and it is
shown here honestly rather than hidden.

In [ ]:
x = 100.0
exact = chebyshev_psi_exact(x)
print(f"ψ({x:.0f}) exact = {exact:.6f}\n")
print(f"{'K tones':>8} {'ψ from tones':>16} {'residual':>12}")
for K in (1, 5, 10, 20, 50):
    approx = chebyshev_psi_explicit(x, zeros_upto(K))
    print(f"{K:>8} {approx:>16.6f} {exact-approx:>12.6f}")

In [ ]:
# The reconstruction across a range — the staircase and its wave approximation
xs = np.linspace(2.0, 60.0, 900)
exact_curve = np.array([chebyshev_psi_exact(t) for t in xs])

fig, ax = plt.subplots(figsize=(10, 4.6))
ax.step(xs, exact_curve, where='post', color='0.15', lw=1.6,
        label='ψ(x) exact  (jumps of ln p)')
for K, c in ((5, '#c44'), (20, '#4a7'), (50, '#47c')):
    ax.plot(xs, [chebyshev_psi_explicit(t, zeros_upto(K)) for t in xs],
            lw=1.1, color=c, alpha=0.9, label=f'{K} tones')
for n, jump in leaf_drops(60):
    ax.axvline(n, color='0.85', lw=0.6, zorder=0)
ax.set_xlabel('x'); ax.set_ylabel('ψ(x)')
ax.set_title('The screw lifts only on primes — and the tones rebuild the staircase')
ax.legend(loc='upper left', frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

---
## 5. The leaf-drop event: the jump height **is** the prime

Not an encoding of it, not a coordinate that constrains it. ψ jumps by exactly
ln p at x = p, so e^{jump} returns the prime with no inversion step.

This is the formal content of Cody's note: *the moment the leaf drops off IS one
of the prime factors.*

In [ ]:
print(f"{'n':>5} {'jump Λ(n)':>12} {'exp(jump)':>12}   reading")
for n, jump in leaf_drops(32):
    kind = 'prime' if abs(math.exp(jump) - n) < 1e-9 else f'prime power of {round(math.exp(jump))}'
    print(f"{n:>5} {jump:>12.6f} {math.exp(jump):>12.4f}   {kind}")

---
## 6. Primes as antinodes — the cymatic picture, read from the other side

PAPER.md §6 establishes the zeros as **node lines** of the zeta field: the still
points, the Chladni sand. The explicit formula reads the *same standing wave*
from the prime side, where the primes are the **antinodes** — the places the
tones stop cancelling and add.

Below: each tone at a given x, and their sum, at a prime versus at a composite.

In [ ]:
def tone_bars(x, ax):
    prof = interference_profile(x, zeros_upto(30))
    gam = [g for g, _ in prof]; val = [t for _, t in prof]
    ax.bar(range(len(gam)), val, color=['#47c' if v >= 0 else '#c44' for v in val])
    ax.axhline(0, color='0.3', lw=0.8)
    ax.set_title(f'x = {x:g}   Σtones = {sum(val):+.3f}', fontsize=10)
    ax.set_xlabel('zero index k'); ax.set_ylabel('tone value')

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for x, ax in zip((31, 32, 33), axes):
    tone_bars(float(x), ax)
plt.suptitle('Tones align at a prime (31) and disperse either side of it', y=1.03)
plt.tight_layout(); plt.show()

---
## 7. RH in the prime domain: one shared envelope

Every tone carries the amplitude 2·x^σ where σ = Re(ρ). On the critical line
that is 2√x — **the same envelope for every zero**.

A single zero at σ > ½ would contribute x^σ and drown every critical-line tone
by a factor x^{σ−½}, which diverges in x. One loud tone and the Chladni figure
has no coherent node structure at all.

    equal envelope  ⟺  all nodes on one line  ⟺  RH

This is the *amplitude* face of the nodal-line argument already in PAPER.md §6 —
the same proof, read in the dual domain, not a second one.

In [ ]:
print(f"{'x':>12} " + ' '.join(f"σ={s:<5.2f}" for s in (0.50, 0.55, 0.60, 0.75)))
for x in (10**2, 10**4, 10**6, 10**9, 10**12):
    row = ' '.join(f"{envelope_ratio(x, s):>7.4g}" for s in (0.50, 0.55, 0.60, 0.75))
    print(f"{x:>12,} {row}")
print()
print("Column σ=0.50 is flat at 1.0 by construction — the shared envelope.")
print("Every other column diverges in x. That divergence is what RH forbids.")

In [ ]:
xs = np.logspace(1, 12, 300)
fig, ax = plt.subplots(figsize=(9, 4.2))
for s, c in ((0.50, '#47c'), (0.55, '#4a7'), (0.60, '#ea3'), (0.75, '#c44')):
    ax.loglog(xs, [envelope_ratio(x, s) for x in xs], color=c, lw=1.6,
              label=f'σ = {s:.2f}')
ax.axhline(1, color='0.4', lw=0.8, ls='--')
ax.set_xlabel('x'); ax.set_ylabel('loudness relative to a critical-line tone')
ax.set_title('An off-line zero drowns the others without bound — RH is the flat line')
ax.legend(frameon=False); plt.tight_layout(); plt.show()

---
## 8. The N-specific leg: ramification is detachment

Twisting by the quadratic character gives ζ_ℚ(√N)(s) = ζ(s)·L(s, χ_N). Every
rational prime then **splits**, is **inert**, or **ramifies** in ℚ(√N), and the
ramified primes are exactly those dividing the discriminant.

For N = p·q squarefree, the ramified primes are exactly p and q — the Euler
factor **degenerates** at precisely the factors. That is the leaf letting go,
written in arithmetic.

**Stated plainly:** detecting ramification by scanning p costs what trial
division costs, and sampling L(s, χ_N) directly costs ~√N — the same wall
Fermat's a²−b² hits. This cell exhibits the exact structure at toy scale. It is
not a shortcut and is not offered as one.

In [ ]:
for N in (33, 91, 143, 1147):
    r = ramified_primes(N, 200)
    print(f"N = {N:>5}   ramified primes ≤ 200: {r}   product = {math.prod(r) if r else None}")
print()
N = 143
print(f"χ_N splitting vector for N = {N} (the cheapest N-specific shadow):")
for p, chi in splitting_vector(N, 60):
    print(f"   p={p:>3}  χ={chi:+d}  {splitting_type(p, N)}")

---
## 9. The shake order

Everything above, assembled: the sequence in which leaves come off the tree up
to x, each with its drop height, alongside the tone reconstruction and the
honest truncation residual.

In [ ]:
s = shake_order(50.0, zeros_upto(50))
print(f"n_tones   = {s['n_tones']}")
print(f"ψ exact   = {s['psi_exact']:.6f}")
print(f"ψ tones   = {s['psi_tones']:.6f}")
print(f"residual  = {s['residual']:+.6f}   (truncation, ~x/K — the resolution wall)")
print()
print("shake order (n, drop height, prime read off the drop):")
for n, j in s['drops']:
    print(f"   {n:>4}   Δψ = {j:.6f}   →  {math.exp(j):.4f}")

---
## 8b. The slot correspondence: which ψ is which

Two ψ live in these repos and they are **different objects** — Chebyshev's ψ
here is a monotone step function on ℝ⁺, one integration above a discrete
measure Λ(n); the ψ in `l_io_photon_path` is a smooth 2D field, two
integrations above a continuous κ. They must stay itemised.

But they are **one slot apart in the same equation**:

    lensing:   L_(I|O)  =  L   −  ψ_Fermat
    primes:    ψ_Cheb   =  x   −  Σ_ρ x^ρ/ρ    (− ln2π − ½ln(1−x⁻²))

    ψ_Cheb      ↔  L_(I|O)     the actual, bent path
    x           ↔  L           the clean geodesic
    Σ_ρ x^ρ/ρ   ↔  ψ_Fermat    the potential — the bend

So Chebyshev ψ is the counterpart of **L_(I|O)**, not of the Fermat potential.
The counterpart of ψ_Fermat is the **zero sum** — which had no name in these
repos until 2026-08-04, because it only ever existed inline inside the
explicit formula. And the main term x **is** L: "the path of least primes",
the phrase the 2026-07-31 primer carries without a formula.

In [ ]:
x = 100.0
d = l_io_decomposition(x, zeros_upto(50))
print(f"  L         = {d['L']:>12.6f}   the clean path — the pole term alone")
print(f"  psi_bend  = {d['psi_bend']:>12.6f}   the zero sum — the Fermat potential's counterpart")
print(f"  trivial   = {d['trivial']:>12.6f}   the trivial-zero tail")
print(f"  L_IO      = {d['L_IO']:>12.6f}   the actual bent path = Chebyshev psi")
print()
print("  identity  L - psi_bend + trivial == L_IO :",
      abs((d['L'] - d['psi_bend'] + d['trivial']) - d['L_IO']) < 1e-12)
print("  agrees with chebyshev_psi_explicit       :",
      abs(d['L_IO'] - chebyshev_psi_explicit(x, zeros_upto(50))) < 1e-12)
print(f"  exact psi({x:.0f}) by sieve                    : {chebyshev_psi_exact(x):.6f}")

In [ ]:
# The bend, drawn: how far the actual path departs from the clean one
xs = np.linspace(2.0, 60.0, 800)
L    = np.array([clean_path_L(t) for t in xs])
bend = np.array([zero_sum(t, zeros_upto(50)) for t in xs])
psi  = np.array([chebyshev_psi_exact(t) for t in xs])

fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                             gridspec_kw={'height_ratios': [2, 1]})
a1.plot(xs, L, color='#47c', lw=1.6, label='L = x   (clean path — least primes)')
a1.step(xs, psi, where='post', color='0.15', lw=1.4,
        label='L_(I|O) = ψ_Cheb   (the actual, bent path)')
a1.set_ylabel('lift'); a1.legend(frameon=False, fontsize=9, loc='upper left')
a1.set_title('The clean path, the bent path, and the potential between them')

a2.plot(xs, bend, color='#c44', lw=1.4, label='ψ_bend = Σ_ρ x^ρ/ρ   (the zero sum)')
a2.axhline(0, color='0.4', lw=0.8)
a2.set_xlabel('x'); a2.set_ylabel('bend'); a2.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

---
## What is open

The construction above is established mathematics assembled onto Cody's screw
axis. The open item is stated once, precisely, and not padded:

**The resolution wall is a *measurement* wall.** Sharply resolving one jump near
x needs zeros to height ~x. Integers do not pay that cost — a winding number is
exact, and the argument principle returns one from a single contour integral
without walking the loop. For N = p·q the double cover ℚ(√N) → ℚ is branched at
exactly p and q, giving two sheets, two strands, B₂ ≅ ℤ: the whole hidden
structure is one integer.

What is not yet written is the **dispersion relation on the zero-divisor
surface** — the hydrocline's own ω(k). That is what fixes the contour and prices
the loop, and it is the next piece of work.

See `ValaQuenta/wiki/archimedes_screw.md` and
`Ainulindale/wiki/83_the_archimedes_screw.md`.